# 33 - Bias Mitigation Strategies - Lab

## Description

This lab implements two mitigation techniques for reducing bias in classification tasks. Starting from a biased lending classifier trained on the Adult Income dataset (loaded from **Lesson 32**), students apply re-weighing and adversarial debiasing, measure fairness and accuracy before and after each technique, and compare the accuracy-fairness trade-offs in a single summary table.

## Objectives

- Implement re-weighing using AIF360 and explain how the computed weights correct the skewed joint distribution of protected attribute and label.
- Implement adversarial debiasing using AIF360, explain the two-model training loop, and describe what the adversary's prediction loss signals to the classifier.
- Produce a side-by-side comparison of re-weighing and adversarial debiasing showing accuracy cost and fairness improvement across two fairness metrics.

## Prerequisites

This lesson's lecture content, plus lesson 32 (Fairness and Bias in AI Systems). Students must understand Demographic Parity, Equalized Odds, and how fairness metrics are computed using AIF360.

## Required Software/Packages

- AIF360 (Algorithmic Fairness 360): `pip install aif360`
- scikit-learn (version 1.0+): included in standard environment
- pandas, numpy, matplotlib: included in standard environment
- TensorFlow/Keras (for adversarial debiasing): included in standard environment

## Environment/Setup Instructions

Run the import cell below to load all required libraries. This lab uses the Adult Income dataset pre-loaded in AIF360.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from aif360.datasets import AdultDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.inprocessing import AdversarialDebiasing
from aif360.metrics import BinaryLabelDatasetMetric
import warnings
warnings.filterwarnings('ignore')

## Background/Context

Bias mitigation techniques operate at different points in the ML pipeline. This lab compares a pre-processing method (re-weighing) and an in-processing method (adversarial debiasing) on a lending classification task. The Adult Income dataset includes gender as a protected attribute; we measure whether male and female applicants receive equal positive-prediction rates (Demographic Parity) before and after mitigation. Accuracy is measured on the original (unweighted, undebiased) test set.

## Exercise Instructions

1. Load the Adult Income dataset from AIF360, split into train/test, train a baseline logistic regression classifier, and compute baseline Demographic Parity difference and accuracy.
2. Apply Reweighing to training data, retrain the logistic regression classifier on reweighted samples, and compute Demographic Parity difference and accuracy on the test set. Record results. 
3. Apply AdversarialDebiasing with the default two-model architecture, train the debiased classifier, and compute Demographic Parity difference and accuracy on the test set. Record results. 
4. Create a comparison table with rows for baseline, re-weighted, and adversarial techniques and columns for DP Difference Before, DP Difference After, Accuracy Before, and Accuracy After. Write one sentence interpreting which technique offers the better fairness-accuracy trade-off. 

## Starter Code

In [ ]:
# Load and prepare the Adult Income dataset
dataset_original = AdultDataset(
    protected_attributes=['sex'],
    privileged_classes=[[1]],  # male=1 is privileged
    categorical_features=[],
    features_to_drop=[]
)

# Split into train and test (70-30 split)
np.random.seed(42)
train_size = int(0.7 * len(dataset_original))
indices = np.random.permutation(len(dataset_original))
train_indices = indices[:train_size]
test_indices = indices[train_size:]

dataset_train = dataset_original.subset(train_indices)
dataset_test = dataset_original.subset(test_indices)

print(f"Training set size: {len(dataset_train)}")
print(f"Test set size: {len(dataset_test)}")
print(f"Protected attribute: {dataset_original.protected_attribute_names}")
print(f"Target variable: {dataset_original.label_names}")

## Step 1: Baseline Classifier 

In [ ]:
# Train baseline logistic regression classifier
X_train = dataset_train.features
y_train = dataset_train.labels.ravel()

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train baseline model
baseline_model = LogisticRegression(random_state=42, max_iter=1000)
baseline_model.fit(X_train_scaled, y_train)

# Make predictions on test set
X_test = dataset_test.features
X_test_scaled = scaler.transform(X_test)
y_test = dataset_test.labels.ravel()

# Create prediction dataset for fairness metrics
dataset_test_pred = dataset_test.copy()
dataset_test_pred.labels = baseline_model.predict(X_test_scaled).reshape(-1, 1)

# Compute baseline fairness metrics
metric_baseline = BinaryLabelDatasetMetric(
    dataset_test_pred,
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}]
)

dp_diff_baseline = abs(metric_baseline.mean_difference())
accuracy_baseline = np.mean(y_test == baseline_model.predict(X_test_scaled))

print(f"Baseline Demographic Parity Difference: {dp_diff_baseline:.4f}")
print(f"Baseline Accuracy: {accuracy_baseline:.4f}")

## Expected Output: Step 1

A Demographic Parity Difference value between 0 and 1 (where 0 = perfect parity, 1 = maximum disparity) and a baseline accuracy value between 0 and 1.

## Step 2: Re-weighing Mitigation

In [ ]:
# Apply Reweighing to the training data
reweighing = Reweighing(
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}]
)

# Fit and transform the training data
dataset_train_reweighted = reweighing.fit_transform(dataset_train)

# Extract reweighted training data
X_train_reweighted = dataset_train_reweighted.features
y_train_reweighted = dataset_train_reweighted.labels.ravel()
sample_weights = dataset_train_reweighted.instance_weights.ravel()

# Standardize reweighted features
X_train_reweighted_scaled = scaler.fit_transform(X_train_reweighted)

# Train logistic regression on reweighted data
reweighted_model = LogisticRegression(random_state=42, max_iter=1000)
reweighted_model.fit(X_train_reweighted_scaled, y_train_reweighted, sample_weight=sample_weights)

# Make predictions on test set
dataset_test_reweighted_pred = dataset_test.copy()
dataset_test_reweighted_pred.labels = reweighted_model.predict(X_test_scaled).reshape(-1, 1)

# Compute fairness metrics for reweighted model
metric_reweighted = BinaryLabelDatasetMetric(
    dataset_test_reweighted_pred,
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}]
)

dp_diff_reweighted = abs(metric_reweighted.mean_difference())
accuracy_reweighted = np.mean(y_test == reweighted_model.predict(X_test_scaled))

print(f"Re-weighted Demographic Parity Difference: {dp_diff_reweighted:.4f}")
print(f"Re-weighted Accuracy: {accuracy_reweighted:.4f}")
print(f"Accuracy Drop from Baseline: {accuracy_baseline - accuracy_reweighted:.4f}")
print(f"Fairness Improvement (DP Diff Reduction): {dp_diff_baseline - dp_diff_reweighted:.4f}")

## Expected Output: Step 2

The re-weighted Demographic Parity Difference (should be less than baseline if mitigation worked), the re-weighted accuracy, and the improvement/cost metrics.

## Step 3: Adversarial Debiasing 

In [ ]:
# Apply AdversarialDebiasing
adversarial_debiaser = AdversarialDebiasing(
    protected_attribute_names=['sex'],
    privileged_groups=[{'sex': 1}],
    unprivileged_groups=[{'sex': 0}],
    scope_name='debiased_classifier',
    sess=None
)

# Fit the debiased model on training data
adversarial_debiaser.fit(dataset_train)

# Make predictions on test set
dataset_test_adversarial_pred = adversarial_debiaser.predict(dataset_test)

# Compute fairness metrics for adversarial debiased model
metric_adversarial = BinaryLabelDatasetMetric(
    dataset_test_adversarial_pred,
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}]
)

dp_diff_adversarial = abs(metric_adversarial.mean_difference())
accuracy_adversarial = np.mean(y_test == dataset_test_adversarial_pred.labels.ravel())

print(f"Adversarial Debiased Demographic Parity Difference: {dp_diff_adversarial:.4f}")
print(f"Adversarial Debiased Accuracy: {accuracy_adversarial:.4f}")
print(f"Accuracy Drop from Baseline: {accuracy_baseline - accuracy_adversarial:.4f}")
print(f"Fairness Improvement (DP Diff Reduction): {dp_diff_baseline - dp_diff_adversarial:.4f}")

## Expected Output: Step 3

The adversarially debiased Demographic Parity Difference (should be less than baseline if mitigation worked), the accuracy, and the improvement/cost metrics.

## Step 4: Comparison Table and Interpretation

In [ ]:
# Create comparison table
comparison_data = {
    'Technique': ['Baseline', 'Re-weighted', 'Adversarial Debiasing'],
    'DP Diff (Before)': [dp_diff_baseline, dp_diff_baseline, dp_diff_baseline],
    'DP Diff (After)': [dp_diff_baseline, dp_diff_reweighted, dp_diff_adversarial],
    'Accuracy (Before)': [accuracy_baseline, accuracy_baseline, accuracy_baseline],
    'Accuracy (After)': [accuracy_baseline, accuracy_reweighted, accuracy_adversarial]
}

comparison_df = pd.DataFrame(comparison_data)
print("\nMitigation Techniques Comparison")
print(comparison_df.to_string(index=False))

# Compute and display trade-off metrics
print("\nTrade-off Analysis")
reweighted_fairness_gain = dp_diff_baseline - dp_diff_reweighted
reweighted_accuracy_loss = accuracy_baseline - accuracy_reweighted
adversarial_fairness_gain = dp_diff_baseline - dp_diff_adversarial
adversarial_accuracy_loss = accuracy_baseline - accuracy_adversarial

print(f"Re-weighing: Fairness gain = {reweighted_fairness_gain:.4f}, Accuracy loss = {reweighted_accuracy_loss:.4f}")
print(f"Adversarial: Fairness gain = {adversarial_fairness_gain:.4f}, Accuracy loss = {adversarial_accuracy_loss:.4f}")

## Expected Output: Step 4

A table with three rows and six columns showing DP Difference and Accuracy before and after mitigation for each technique. Below the table, fairness gain and accuracy loss for each mitigation technique.

## Questions/Tasks

1. In the re-weighing step, which group (privileged or unprivileged) received higher sample weights, and why does this make sense for correcting the Demographic Parity difference?
2. In adversarial debiasing, what role does the adversary play, and how does maximizing the adversary's loss signal guide the classifier toward learning fairer representations?

## Optional Challenges

**Challenge 1:** Compute and report Equalized Odds difference (equal TPR and FPR across groups) for each technique in addition to Demographic Parity. Which fairness metric improved more?

**Challenge 2:** Modify the hyperparameter λ (fairness-accuracy weight) in adversarial debiasing from 0.1 to 2.0 and plot how the Demographic Parity difference and accuracy change. What trade-off shape emerges?